In [1]:
import dlib
print("CUDA enabled:", dlib.DLIB_USE_CUDA)
print("GPU count:", dlib.cuda.get_num_devices())

CUDA enabled: True
GPU count: 1


In [9]:
import face_recognition
import os
import numpy as np

known_encodings = []
known_names = []

faces_dir = "known_faces"

for filename in os.listdir(faces_dir):
    if filename.endswith(".jpg") or filename.endswith(".png"):
        
        path = os.path.join(faces_dir, filename)

        image = face_recognition.load_image_file(path)
        encodings = face_recognition.face_encodings(image)

        if len(encodings) > 0:
            known_encodings.append(encodings[0])

            name = os.path.splitext(filename)[0]  # remove extension
            known_names.append(name)

print("Loaded faces:", known_names)

import threading
import pygame
import cv2
import face_recognition
import time
def play_sound(file):
    """Play sound asynchronously"""
    threading.Thread(target=_play, args=(file,), daemon=True).start()

def _play(file):
    pygame.mixer.music.load(file)
    pygame.mixer.music.play()

Loaded faces: ['dad', 'me']


In [14]:
from collections import Counter

pygame.mixer.init()
cap = cv2.VideoCapture(0)

# -------------------------
# Timestamp for throttling
# -------------------------
last_play_time = 0
MIN_INTERVAL = 5  # seconds
name_buffer = []

while True:
    ret, frame = cap.read()
    blurred = cv2.GaussianBlur(frame, (45, 45), 0)
    # frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if not ret:
        break

    # Resize for speed
    small_frame = cv2.resize(frame, (0,0), fx=0.25, fy=0.25)
    rgb_small = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    # GPU-enabled detection
    face_locations = face_recognition.face_locations(rgb_small, model="cnn")
    encodings = face_recognition.face_encodings(rgb_small, face_locations)

    # Scale locations back to original frame
    face_locations = [(top*4, right*4, bottom*4, left*4) for (top, right, bottom, left) in face_locations]

    for (top, right, bottom, left), encoding in zip(face_locations, encodings):
        # Compare with known faces
        distances = face_recognition.face_distance(known_encodings, encoding)
        best_match = np.argmin(distances)
        if(1-distances[best_match] > 0.6):
            name = known_names[best_match]
        else:
            name = "Unknown"
        name_buffer.append(name)
        
        if len(name_buffer) > 20:
            # Count occurrences
            counts = Counter(name_buffer)
            # Find the most common element
            normalized_name, count = counts.most_common(1)[0]
            
            # Only play sound if enough time has passed
            current_time = time.time()
            if current_time - last_play_time >= MIN_INTERVAL:
                if (normalized_name == "Unknown"):
                    play_sound("audio/stranger.mp3")
                else:
                    play_sound("audio/home.mp3")
                last_play_time = current_time
            name_buffer = []

        # Draw box + name
        cv2.rectangle(blurred, (left, top), (right, bottom), (0,255,0), 2)
        cv2.putText(blurred, f"{name} ({1-distances[best_match]:.2f})", 
                    (left, top-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    
    cv2.imshow("Face Recognition", blurred)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
pygame.mixer.quit()

In [5]:
!pip list

Package                   Version
------------------------- -----------
anyio                     4.12.1
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.1
async-lru                 2.2.0
attrs                     25.4.0
babel                     2.18.0
beautifulsoup4            4.14.3
bleach                    6.3.0
certifi                   2026.2.25
cffi                      2.0.0
charset-normalizer        3.4.5
click                     8.3.1
cmake                     4.2.3
colorama                  0.4.6
comm                      0.2.3
debugpy                   1.8.20
decorator                 5.2.1
defusedxml                0.7.1
dlib                      20.0.99
exceptiongroup            1.3.1
executing                 2.2.1
face-recognition          1.3.0
face-recognition-models   0.3.0
fastjsonschema            2.21.2
fqdn                      1.5.1
h11                       0.16.0
httpcore         


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
